# T06 — Train normal-only anomaly baseline

Upload `t06_training_payload_v2.zip` to the root of Google Drive, then run every cell. A CPU runtime is sufficient; Isolation Forest does not use a GPU. V2 excludes pose-quality signals from the model and emits `insufficient_evidence` for low-quality windows.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Change this only if you uploaded the payload elsewhere.
PAYLOAD = '/content/drive/MyDrive/t06_training_payload_v2.zip'
PROJECT_ROOT = '/content/ExhibitFlow'
import os
assert os.path.isfile(PAYLOAD), f'Payload not found: {PAYLOAD}'
print('payload:', PAYLOAD, f'({os.path.getsize(PAYLOAD)/1e6:.1f} MB)')

In [ ]:
!rm -rf {PROJECT_ROOT}
!mkdir -p {PROJECT_ROOT}
!unzip -q -o "{PAYLOAD}" -d {PROJECT_ROOT}
REPO = f'{PROJECT_ROOT}/Museum/pose_rule_anomaly'
%cd {REPO}
!find . -maxdepth 3 -type f -print
!ls -lh outputs/windows.npz

In [ ]:
# Colab normally includes these packages. Install only if an import is missing.
import numpy, sklearn, joblib, matplotlib
print('numpy', numpy.__version__)
print('scikit-learn', sklearn.__version__)

In [ ]:
# This cell checks shapes, sources, split, and camera balancing without fitting.
!python -u -m src.anomaly.train_baseline --validate-only

In [ ]:
# Actual T06 training. Expected to be CPU-bound.
!python -u -m src.anomaly.train_baseline --trees 300 --max-train-per-camera 2500

In [ ]:
# Verify outputs before download.
import json, pathlib
out = pathlib.Path(REPO) / 'outputs' / 'anomaly'
required = ['isolation_forest.joblib', 'anomaly_scores.jsonl', 'score_distribution.png', 'training_report.md']
for name in required:
    path = out / name
    assert path.is_file() and path.stat().st_size > 0, f'Missing output: {path}'
    print(name, f'{path.stat().st_size/1e6:.2f} MB')
with (out / 'anomaly_scores.jsonl').open() as f:
    rows = sum(1 for _ in f)
assert rows == 9563, f'Expected 9563 scores, got {rows}'
print('score rows:', rows)
print((out / 'training_report.md').read_text())

In [ ]:
RESULT_ZIP = '/content/t06_results.zip'
!cd {REPO} && zip -q -r {RESULT_ZIP} outputs/anomaly
from google.colab import files
files.download(RESULT_ZIP)